# broadcasting-rules — faded example 3: Numerically-stable row softmax via keepdim broadcast

> Practice drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `broadcasting-rules`. The last cell reports your progress on the `Numpy: Vectorization and broadcasting` subtopic back to Delta Drills.

**Most of the code is already written — complete the one blanked step**, run the test to check it, then run the last cell to record your progress.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Vectorization and broadcasting` subtopic.
You can copy the token from your Delta Drills account page.

This drill exercises the **atom `broadcasting-rules`**, which bridges to the bank subtopic `Numpy: Vectorization and broadcasting` for EWMA state. Completing all 5 exercises triggers a single `arena-rating` beacon at the end of the notebook.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcasting-rules"
DD_SUBTOPIC = "Numpy: Vectorization and broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

# Track which exercises passed in this session.
_dd_passed = set()

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Row-wise softmax needs two broadcasts back against an `(N, D)` matrix: subtract the per-row max (for numerical stability) and divide by the per-row sum of exponentials. Reducing with `keepdim=True` gives `(N, 1)` results that column-broadcast across the `D` features without any manual `unsqueeze`.

## Faded exercise 3

### Faded — stable row softmax

Implement `row_softmax(x)`: given logits `x` of shape `(N, D)`, return softmax over the **last axis**, shape `(N, D)`, each row summing to 1. Subtract the per-row max before exponentiating for numerical stability.

The scaffold subtracts the per-row max, exponentiates, then normalizes. Fill in the normalization denominator so it divides each row by that row's sum of exponentials.

**Your task:** complete the one blanked step in the code cell below. The surrounding code, function signatures, and variable names are given — work out the missing expression yourself, then run the test.

In [ ]:
def row_softmax(x):
    shifted = x - x.max(dim=1, keepdim=True).values   # (N, D)
    exps = t.exp(shifted)                             # (N, D)
    denom = ____  # TODO: fill in this step — read the prompt cell above
    return exps / denom


def _test():
    t.manual_seed(0)
    x = t.randn(6, 5) * 3.0
    out = row_softmax(x)
    assert out.shape == (6, 5), out.shape
    # Each row is a valid probability distribution.
    assert t.allclose(out.sum(dim=1), t.ones(6), atol=1e-6)
    assert (out >= 0).all()
    # Matches torch's reference softmax.
    ref = t.softmax(x, dim=1)
    assert t.allclose(out, ref, atol=1e-6)


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report your progress

Run the cell below to send your progress to Delta Drills. It only counts if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',  # single-exercise standalone — neutral signal
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def row_softmax(x):
    shifted = x - x.max(dim=1, keepdim=True).values   # (N, D)
    exps = t.exp(shifted)                             # (N, D)
    denom = exps.sum(dim=1, keepdim=True)             # (N, 1)
    return exps / denom
```
</details>